# TinyForge d12: 1-hour T4 test

One Colab T4, about 60 minutes of training, then the runtime disconnects itself.

**How to run:** Runtime > Change runtime type > **T4 GPU**, then **Runtime > Run all**. Approve the Google Drive prompt in the first cell. Nothing else needs a click.

What it does:
1. Mounts Drive and clones `t4-kaggle-d12`. Starts a safety watchdog that disconnects the runtime after `WATCHDOG_MIN` minutes no matter what.
2. Builds the fp16 trainer with the runtime's own nvcc (CUDA 12 or 13). It checks for cuBLAS and installs the pip wheel if it is missing. If the build can't work, it fails with a clear message.
3. Data: a 32768-token tokenizer, about 90M ClimbMix train tokens, and 262,144 val tokens. These are cached in Drive at `MyDrive/TinyForge/data/`, so the first run builds them (about 10 min) and later runs copy them (under 1 min).
4. Trains d12 with `-b 16 -t 512 -d 65536 -x 1100`, about 72M tokens and about 57 min at 21k tok/s. The LR schedule finishes inside the hour. `--time-limit` is a backstop. Val loss and bpb are checked every 100 steps. A checkpoint is written every 200 steps and mirrored to `MyDrive/TinyForge/runs/<RUN_NAME>/`. Re-running resumes from the latest Drive checkpoint.
5. Writes a summary: tokens, steady tok/s, final train and val loss, bpb, and wall time. Exports a Hugging Face folder to Drive and prints a short sample.
6. **Last cell**: saves everything to Drive, flushes Drive, and calls `runtime.unassign()`.

Every stage is wrapped, so an error in one stage skips the stages that depend on it. Run all still reaches the final save-and-disconnect cell.

In [ ]:
# ---- 1. Setup: config, Drive, watchdog, GPU check, clone ----
import os, sys, re, glob, json, time, shutil, signal, subprocess, threading, traceback, statistics, importlib.util

# Knobs
REPO_URL     = "https://github.com/michaelp91-dev/tinyforge"
BRANCH       = "t4-kaggle-d12"
RUN_NAME     = "d12_1hr"      # Drive run folder; re-running with the same name resumes it
FRESH_RUN    = False          # True = start a new run folder (RUN_NAME + timestamp)
MAX_STEPS    = 1100           # 1100 x 65,536 = 72.1M tokens, about 57 min at 3.11 s/step
TIME_LIMIT_S = 3900           # trainer backstop: checkpoints and exits at 65 min of training
VAL_EVERY    = 100
VAL_BATCHES  = 31             # 31 x 8192 = the whole 262,144-token val set
CKPT_EVERY   = 200
TRAIN_TOKENS = 90_000_000
VAL_TOKENS   = 262_144
VOCAB        = 32768
TOK_CHARS    = 50_000_000     # characters used to train the tokenizer
WATCHDOG_MIN = 120            # hard stop: disconnect the runtime after this many minutes
DO_SAMPLE    = True
AUTO_DISCONNECT = True        # last cell calls runtime.unassign()

LOCAL_TEST = os.environ.get("TF_LOCAL_TEST") == "1"   # box-side dry run only; ignore in Colab
CONTENT    = os.environ.get("TF_CONTENT", "/content")
T0 = time.time()
STATUS = {}

def banner(msg):
    print("\n" + "!" * 78 + "\n" + msg + "\n" + "!" * 78, flush=True)

def run_stage(name, fn, needs=()):
    """Run one notebook stage; never raise, so Run all reaches the last cell."""
    bad = [n for n in needs if STATUS.get(n, {}).get("ok") is not True]
    if bad:
        STATUS[name] = {"ok": False, "skipped": True, "error": f"skipped: {', '.join(bad)} did not succeed"}
        banner(f"[{name}] SKIPPED because {', '.join(bad)} failed (see above).")
        return None
    t = time.time()
    try:
        out = fn()
        STATUS[name] = {"ok": True, "secs": round(time.time() - t, 1)}
        print(f"[{name}] ok in {time.time() - t:.0f}s", flush=True)
        return out
    except BaseException as e:   # KeyboardInterrupt too: still let the last cell run
        STATUS[name] = {"ok": False, "secs": round(time.time() - t, 1), "error": f"{type(e).__name__}: {e}"}
        banner(f"[{name}] FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        return None

def sh(cmd, check=True, env=None, cwd=None):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, env=env, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{(r.stdout + r.stderr)[-3000:]}")
    return r

def _setup():
    global DRIVE_ROOT, REPO, RUN_NAME, GPU_CC, GPU_NAME
    if LOCAL_TEST:
        DRIVE_ROOT = os.environ["TF_DRIVE_ROOT"]
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = "/content/drive/MyDrive/TinyForge"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if FRESH_RUN:
        RUN_NAME = RUN_NAME + time.strftime("_%Y%m%d_%H%M")
    print("Drive folder:", DRIVE_ROOT, "| run:", RUN_NAME)

    if LOCAL_TEST:
        GPU_NAME, GPU_CC = os.environ.get("TF_GPU_NAME", "Tesla T4"), os.environ.get("TF_GPU_CC", "7.5")
    else:
        r = sh("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader", check=False)
        if r.returncode != 0 or not r.stdout.strip():
            raise RuntimeError("No GPU visible. Runtime > Change runtime type > T4 GPU, then Run all again.")
        GPU_NAME, GPU_CC, mem = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
        print(f"GPU: {GPU_NAME}, compute capability {GPU_CC}, {mem}")
        print(sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader", check=False).stdout.strip(), "(driver)")
        if "T4" not in GPU_NAME:
            print(f"NOTE: this notebook is tuned for a T4; on {GPU_NAME} it builds for sm_{GPU_CC.replace('.', '')} and step times will differ.")

    REPO = os.path.join(CONTENT, "tinyforge")
    if os.path.isdir(os.path.join(REPO, ".git")):
        sh(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH])
        sh(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"])
    else:
        sh(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
    head = sh(["git", "-C", REPO, "log", "-1", "--format=%h %s"]).stdout.strip()
    STATUS["commit"] = head
    print("repo:", REPO, "@", head)

def _watchdog():
    deadline = T0 + WATCHDOG_MIN * 60
    while time.time() < deadline:
        time.sleep(30)
    banner(f"WATCHDOG: {WATCHDOG_MIN} min reached, saving logs and disconnecting the runtime.")
    try:
        for p in glob.glob(os.path.join(CONTENT, "runs", RUN_NAME, "*.log")):
            shutil.copy(p, os.path.join(DRIVE_ROOT, "runs", RUN_NAME, os.path.basename(p)))
    except Exception as e:
        print("watchdog copy failed:", e)
    if not LOCAL_TEST:
        try:
            from google.colab import drive; drive.flush_and_unmount()
        except Exception:
            pass
        from google.colab import runtime; runtime.unassign()

run_stage("setup", _setup)
if not LOCAL_TEST:
    threading.Thread(target=_watchdog, daemon=True).start()
    print(f"watchdog armed: runtime disconnects at {WATCHDOG_MIN} min no matter what")

In [ ]:
# ---- 2. Build the fp16 T4 trainer with this runtime's CUDA ----
TRAIN_ENV = dict(os.environ)
BIN = None

def _find_nvcc():
    cands = [shutil.which("nvcc"), "/usr/local/cuda/bin/nvcc"] + sorted(glob.glob("/usr/local/cuda-*/bin/nvcc"), reverse=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise RuntimeError("nvcc not found (looked on PATH, /usr/local/cuda/bin, /usr/local/cuda-*/bin). "
                       "This runtime has no CUDA toolkit; switch to a GPU runtime (T4) and Run all again.")

def _pip_cublas(major):
    """Install the cuBLAS wheel matching the toolkit's major version; return its (include, lib) dirs."""
    pkg = "nvidia-cublas-cu12" if major == 12 else f"nvidia-cublas>={major},<{major + 1}"
    print("installing", pkg, "...")
    sh([sys.executable, "-m", "pip", "install", "-q", pkg])
    import importlib, site
    roots = site.getsitepackages() + [site.getusersitepackages()]
    for root in roots:
        for sub in (f"nvidia/cu{major}", "nvidia/cublas"):
            d = os.path.join(root, sub)
            libs = glob.glob(os.path.join(d, "lib", "libcublas.so*"))
            if libs:
                lib = os.path.join(d, "lib")
                for name in ("libcublas", "libcublasLt"):   # make -lcublas resolve
                    real = sorted(glob.glob(os.path.join(lib, name + ".so.*")))
                    if real and not os.path.exists(os.path.join(lib, name + ".so")):
                        os.symlink(os.path.basename(real[0]), os.path.join(lib, name + ".so"))
                return os.path.join(d, "include"), lib
    raise RuntimeError(f"pip installed {pkg} but no libcublas.so was found under {roots}")

def _missing_libs(binary, env):
    r = sh(["ldd", binary], check=False, env=env)
    return [l.split()[0] for l in r.stdout.splitlines() if "not found" in l]

def _build():
    global BIN, TRAIN_ENV
    nvcc = _find_nvcc()
    ver = sh([nvcc, "--version"]).stdout
    m = re.search(r"release (\d+)\.(\d+)", ver)
    major = int(m.group(1)) if m else 0
    print(f"nvcc: {nvcc} (CUDA {m.group(1)}.{m.group(2)})" if m else f"nvcc: {nvcc}\n{ver}")
    if major and major < 11:
        raise RuntimeError(f"CUDA {major} is too old for this trainer (needs 11.8+).")
    cuda_home = os.path.dirname(os.path.dirname(os.path.realpath(nvcc)))
    env = dict(os.environ)
    env["PATH"] = os.path.dirname(nvcc) + ":" + env.get("PATH", "")
    make = ["make", "-C", REPO, "train_tinyforge", "GPU=t4"]
    cc = GPU_CC.replace(".", "")
    if cc != "75":   # not a T4: same fp16 code path, compiled for this GPU
        make.append(f"NVCC_ARCH=-gencode arch=compute_{cc},code=sm_{cc} -DENABLE_FP16 -DTF_FP16_GEMMEX")
    extra = os.environ.get("TF_NVCC_EXTRA", "")   # box test hook; empty in Colab
    lib_dirs = [d for d in (os.path.join(cuda_home, "lib64"), os.path.join(cuda_home, "lib"),
                            os.path.join(cuda_home, "targets/x86_64-linux/lib")) if os.path.isdir(d)]
    log = os.path.join(CONTENT, "build.log")
    for attempt in (1, 2):
        nv = f"{nvcc} {extra}".strip()
        sh(["rm", "-f", os.path.join(REPO, "train_tinyforge")])
        r = subprocess.run(make + [f"NVCC={nv}"], capture_output=True, text=True, env=env)
        open(log, "a").write(f"=== attempt {attempt}: {' '.join(make)} NVCC={nv}\n{r.stdout}{r.stderr}\n")
        if r.returncode == 0:
            break
        out = r.stdout + r.stderr
        print(out[-2500:])
        if attempt == 1 and re.search(r"cublas|cannot find -l", out, re.I):
            inc, lib = _pip_cublas(major)
            extra = f"{extra} -I{inc} -L{lib}".strip()
            lib_dirs.insert(0, lib)
            continue
        raise RuntimeError(f"BUILD FAILED with CUDA {major} (log: {log}). The compiler error is printed above.")
    BIN = os.path.join(REPO, "train_tinyforge")

    # Every shared library must resolve at run time.
    TRAIN_ENV = dict(env)
    TRAIN_ENV["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [env.get("LD_LIBRARY_PATH", "")]).strip(":")
    miss = _missing_libs(BIN, TRAIN_ENV)
    if miss:
        extra_dirs = sorted({os.path.dirname(p) for p in glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib/*.so*")})
        TRAIN_ENV["LD_LIBRARY_PATH"] += ":" + ":".join(extra_dirs)
        miss = _missing_libs(BIN, TRAIN_ENV)
    if miss and any("cublas" in x for x in miss):
        _, lib = _pip_cublas(major)
        TRAIN_ENV["LD_LIBRARY_PATH"] = lib + ":" + TRAIN_ENV["LD_LIBRARY_PATH"]
        miss = _missing_libs(BIN, TRAIN_ENV)
    if miss:
        raise RuntimeError(f"built OK but these libraries can't be found at run time: {miss}")
    for l in sh(["ldd", BIN], env=TRAIN_ENV).stdout.splitlines():
        if "cublas" in l or "cudart" in l:
            print("  " + l.strip())
    print("built", BIN)

run_stage("build", _build, needs=["setup"])

In [ ]:
# ---- 3. Data: tokenizer + ~90M train / 262,144 val tokens, cached in Drive ----
DATA = os.path.join(CONTENT, "data")
TOK_DIR, MIX_DIR = os.path.join(DATA, "tokenizer"), os.path.join(DATA, "climbmix")

def _ntok(path):
    import numpy as np
    return int(np.fromfile(path, dtype=np.int32, count=3)[2])

def _data_ok(root):
    t, c = os.path.join(root, "tokenizer"), os.path.join(root, "climbmix")
    need = [os.path.join(t, "tokenizer.json"), os.path.join(t, "token_bytes.bin"), os.path.join(c, "val.bin")]
    if not all(os.path.exists(p) for p in need):
        return False
    shards = glob.glob(os.path.join(c, "train_*.bin"))
    return bool(shards) and sum(_ntok(s) for s in shards) >= int(TRAIN_TOKENS * 0.95)

def _data():
    cache = os.path.join(DRIVE_ROOT, "data", f"climbmix_v{VOCAB}_{TRAIN_TOKENS // 1_000_000}M")
    if os.path.exists(os.path.join(cache, "COMPLETE")) and _data_ok(cache):
        print("copying cached data from Drive:", cache)
        t = time.time()
        shutil.rmtree(DATA, ignore_errors=True)
        shutil.copytree(cache, DATA, ignore=shutil.ignore_patterns("COMPLETE"))
        print(f"  copied in {time.time() - t:.0f}s")
    elif _data_ok(DATA):
        print("local data already present:", DATA)
    else:
        need = [m for m in ("datasets", "transformers", "tokenizers") if importlib.util.find_spec(m) is None]
        if need:
            sh([sys.executable, "-m", "pip", "install", "-q"] + need)
        py = sys.executable
        t = time.time()
        if not os.path.exists(os.path.join(TOK_DIR, "token_bytes.bin")):
            print("training tokenizer ...", flush=True)
            subprocess.run([py, "python/train_tokenizer.py", "--out", TOK_DIR, "--vocab-size", str(VOCAB),
                            "--max-chars", str(TOK_CHARS)], cwd=REPO, check=True)
        print("tokenizing ClimbMix ...", flush=True)
        subprocess.run([py, "python/prepare_climbmix.py", "--tokenizer", TOK_DIR, "--out", MIX_DIR,
                        "--target-tokens", str(TRAIN_TOKENS), "--val-tokens", str(VAL_TOKENS)], cwd=REPO, check=True)
        if not _data_ok(DATA):
            raise RuntimeError("data prep finished but the files are incomplete")
        print(f"data built in {time.time() - t:.0f}s; caching to Drive ...")
        try:
            shutil.rmtree(cache, ignore_errors=True)
            shutil.copytree(DATA, cache)
            open(os.path.join(cache, "COMPLETE"), "w").write(time.ctime())
        except Exception as e:
            print("WARNING: could not cache data to Drive (training continues):", e)
    shards = sorted(glob.glob(os.path.join(MIX_DIR, "train_*.bin")))
    ntrain, nval = sum(_ntok(s) for s in shards), _ntok(os.path.join(MIX_DIR, "val.bin"))
    print(f"train tokens: {ntrain:,} in {len(shards)} shard(s) | val tokens: {nval:,}")
    if nval < VAL_TOKENS * 0.99:
        raise RuntimeError(f"val.bin has only {nval} tokens")
    if ntrain < MAX_STEPS * 65536:
        print(f"NOTE: {MAX_STEPS} steps need {MAX_STEPS * 65536:,} tokens; the loader will wrap around.")

run_stage("data", _data, needs=["setup"])

In [ ]:
# ---- 4. Train (~57 min). Checkpoints mirror to Drive; re-running resumes ----
RUN_LOCAL = None
RUN_DRIVE = None
TRAIN_LOG = None
TRAIN_WALL = None

def _done_steps(d):
    return sorted(int(os.path.basename(p)[5:]) for p in glob.glob(os.path.join(d, "DONE_*")))

def _sync(final=False):
    """Copy the newest complete checkpoint and the log to Drive; keep one checkpoint there."""
    global _synced
    try:
        if os.path.exists(TRAIN_LOG):
            shutil.copy(TRAIN_LOG, os.path.join(RUN_DRIVE, os.path.basename(TRAIN_LOG)))
        steps = _done_steps(RUN_LOCAL)
        if not steps or steps[-1] <= _synced:
            return
        s = steps[-1]
        for kind in ("model", "state"):
            src = os.path.join(RUN_LOCAL, f"{kind}_{s:08d}.bin")
            dst = os.path.join(RUN_DRIVE, f"{kind}_{s:08d}.bin")
            shutil.copy(src, dst + ".tmp"); os.replace(dst + ".tmp", dst)
        open(os.path.join(RUN_DRIVE, f"DONE_{s:08d}"), "w").close()
        for old in _done_steps(RUN_DRIVE):
            if old != s:
                for f in (f"DONE_{old:08d}", f"model_{old:08d}.bin", f"state_{old:08d}.bin"):
                    try: os.remove(os.path.join(RUN_DRIVE, f))
                    except FileNotFoundError: pass
        _synced = s
        print(f"  [drive] checkpoint step {s} saved to {RUN_DRIVE}", flush=True)
    except Exception as e:
        print("  [drive] sync failed (will retry):", e, flush=True)

def _train():
    global RUN_LOCAL, RUN_DRIVE, TRAIN_LOG, TRAIN_WALL, _synced
    RUN_LOCAL = os.path.join(CONTENT, "runs", RUN_NAME)
    RUN_DRIVE = os.path.join(DRIVE_ROOT, "runs", RUN_NAME)
    os.makedirs(RUN_LOCAL, exist_ok=True); os.makedirs(RUN_DRIVE, exist_ok=True)
    TRAIN_LOG = os.path.join(RUN_LOCAL, "train.log")
    if os.path.exists(os.path.join(RUN_DRIVE, "train.log")) and not os.path.exists(TRAIN_LOG):
        shutil.copy(os.path.join(RUN_DRIVE, "train.log"), TRAIN_LOG)   # keep one log across resumes
    _synced = max(_done_steps(RUN_DRIVE) or [0])
    local = max(_done_steps(RUN_LOCAL) or [0])
    if _synced > local:
        print(f"resuming from Drive checkpoint step {_synced}: copying it local ...", flush=True)
        for f in (f"model_{_synced:08d}.bin", f"state_{_synced:08d}.bin", f"DONE_{_synced:08d}"):
            shutil.copy(os.path.join(RUN_DRIVE, f), os.path.join(RUN_LOCAL, f))
    elif local:
        print(f"resuming from local checkpoint step {local}")
    else:
        print("fresh run")

    cmd = [BIN, "-i", os.path.join(MIX_DIR, "train_*.bin"), "-j", os.path.join(MIX_DIR, "val.bin"),
           "-e", "d12", "-o", RUN_LOCAL, "-b", "16", "-t", "512", "-d", "65536",
           "-x", str(MAX_STEPS), "-v", str(VAL_EVERY), "-m", str(VAL_BATCHES),
           "-n", str(CKPT_EVERY), "-y", "1", "--keep-ckpts", "2", "--time-limit", str(TIME_LIMIT_S),
           "--vocab", str(VOCAB), "--token-bytes", os.path.join(TOK_DIR, "token_bytes.bin")]
    if os.environ.get("TF_TRAIN_CMD"):          # box test hook; unused in Colab
        cmd = json.loads(os.environ["TF_TRAIN_CMD"])
    print(" ".join(cmd), flush=True)

    stop = threading.Event()
    def syncer():
        while not stop.wait(30):
            _sync()
    th = threading.Thread(target=syncer, daemon=True); th.start()

    t = time.time()
    rc = None
    with open(TRAIN_LOG, "a") as logf:
        logf.write(f"\n=== session {time.strftime('%Y-%m-%d %H:%M:%S')}  commit {STATUS.get('commit')}  gpu {GPU_NAME}\n")
        logf.write(" ".join(cmd) + "\n"); logf.flush()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=TRAIN_ENV)
        try:
            for line in p.stdout:
                logf.write(line); logf.flush()
                m = re.match(r"step (\d+)/", line)
                if not m or int(m.group(1)) % 25 == 0 or int(m.group(1)) <= 3:
                    print(line, end="", flush=True)
            rc = p.wait()
        finally:
            if p.poll() is None:            # cell interrupted: let the trainer stop cleanly
                p.send_signal(signal.SIGINT); time.sleep(5); p.kill()
            TRAIN_WALL = time.time() - t
            stop.set(); th.join(timeout=5)
            logf.write(f"=== trainer exit {rc} after {TRAIN_WALL:.0f}s\n")
    _sync(final=True)
    print(f"training wall time this session: {TRAIN_WALL / 60:.1f} min, exit code {rc}")
    if rc != 0:
        tail = "".join(open(TRAIN_LOG).readlines()[-25:])
        raise RuntimeError(f"trainer exited with code {rc}. Last lines:\n{tail}")

run_stage("train", _train, needs=["build", "data"])

In [ ]:
# ---- 5. Summary ----
SUMMARY = {}

def _summary():
    global SUMMARY
    lines = open(TRAIN_LOG).read().splitlines()
    if not TRAIN_LOG or not os.path.exists(TRAIN_LOG):
        raise RuntimeError("no training log (training did not start)")
    steps, sess_first = [], 0
    for l in lines:
        if l.startswith("=== session"):
            sess_first = 0
        m = re.match(r"step (\d+)/(\d+)\s+loss ([\d.naninf]+)\s+lr_mult ([\d.]+)\s+([\d.]+)s\s+(\d+) tok/s", l)
        if m:
            sess_first += 1
            steps.append(dict(step=int(m.group(1)), loss=float(m.group(3)), secs=float(m.group(5)),
                              tps=int(m.group(6)), warm=sess_first > 5))
    vals = [(float(a), float(b)) for a, b in re.findall(r"val loss ([\d.]+)\s+bpb ([\d.]+)", "\n".join(lines))]
    if not steps:
        raise RuntimeError("no step lines in the log")
    last = steps[-1]["step"]
    steady = [s["tps"] for s in steps if s["warm"]] or [s["tps"] for s in steps]
    tail = [s["loss"] for s in steps[-20:]]
    SUMMARY = {
        "run": RUN_NAME, "commit": STATUS.get("commit"), "gpu": GPU_NAME,
        "steps_done": last, "max_steps": MAX_STEPS,
        "tokens_processed": last * 65536,
        "steady_tok_per_s_median": int(statistics.median(steady)),
        "steady_s_per_step_median": round(statistics.median([s["secs"] for s in steps if s["warm"]] or [s["secs"] for s in steps]), 3),
        "final_train_loss": steps[-1]["loss"],
        "final_train_loss_avg20": round(sum(tail) / len(tail), 4),
        "final_val_loss": vals[-1][0] if vals else None,
        "final_val_bpb": vals[-1][1] if vals else None,
        "val_curve": vals,
        "fp16_overflow_skips": sum("gradient overflow" in l for l in lines),
        "self_checks": [l.strip() for l in lines if "check" in l.lower() and ("gemm" in l.lower() or "fused" in l.lower())][:4],
        "train_wall_min_this_session": round(TRAIN_WALL / 60, 1) if TRAIN_WALL else None,
        "notebook_wall_min": round((time.time() - T0) / 60, 1),
        "stage_secs": {k: v.get("secs") for k, v in STATUS.items() if isinstance(v, dict)},
        "finished_lr_schedule": last >= MAX_STEPS,
    }
    txt = "\n".join(f"{k}: {v}" for k, v in SUMMARY.items())
    print(txt)
    open(os.path.join(RUN_LOCAL, "summary.txt"), "w").write(txt + "\n")
    json.dump(SUMMARY, open(os.path.join(RUN_LOCAL, "summary.json"), "w"), indent=1)
    for f in ("summary.txt", "summary.json"):
        shutil.copy(os.path.join(RUN_LOCAL, f), os.path.join(RUN_DRIVE, f))

run_stage("summary", _summary, needs=["setup"])

In [ ]:
# ---- 6. Export to Hugging Face format (Drive) + a short sample ----
def _export():
    s = max(_done_steps(RUN_LOCAL) or [0]) if RUN_LOCAL else 0
    if not s:
        raise RuntimeError("no finished checkpoint to export")
    ckpt = os.path.join(RUN_LOCAL, f"model_{s:08d}.bin")
    out = os.path.join(CONTENT, "export", f"{RUN_NAME}-step{s}-hf")
    shutil.rmtree(out, ignore_errors=True)
    subprocess.run([sys.executable, "python/export_hf.py", "--input", ckpt, "--output", out, "--tokenizer", TOK_DIR],
                   cwd=REPO, check=True)
    dst = os.path.join(DRIVE_ROOT, "exports", os.path.basename(out))
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(out, dst)
    print("HF export:", dst)
    if not DO_SAMPLE:
        return
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(out, trust_remote_code=True)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        model = AutoModelForCausalLM.from_pretrained(out, trust_remote_code=True).to(dev).eval()
        g = torch.Generator(device=dev).manual_seed(0)
        samples = []
        for prompt in ("The capital of France is", "Once upon a time,", "In this article we explain how"):
            ids = torch.tensor([[tok.bos_token_id] + tok.encode(prompt, add_special_tokens=False)], device=dev)
            with torch.no_grad():
                for _ in range(48):
                    logits = model(input_ids=ids[:, -512:]).logits[:, -1, :].float() / 0.8
                    v, _ = torch.topk(logits, 40)
                    logits[logits < v[:, -1:]] = -float("inf")
                    nxt = torch.multinomial(torch.softmax(logits, -1), 1, generator=g)
                    ids = torch.cat([ids, nxt], 1)
            text = tok.decode(ids[0, 1:].tolist())
            samples.append(text)
            print("---", text.replace("\n", " "))
        open(os.path.join(RUN_DRIVE, "samples.txt"), "w").write("\n---\n".join(samples) + "\n")
        del model
    except Exception as e:
        print("sample generation skipped:", type(e).__name__, e)

run_stage("export", _export, needs=["setup"])

In [ ]:
# ---- 7. Save everything to Drive, then disconnect the runtime ----
def _final_save():
    rd = os.path.join(DRIVE_ROOT, "runs", RUN_NAME)
    os.makedirs(rd, exist_ok=True)
    if TRAIN_LOG and os.path.exists(TRAIN_LOG):
        shutil.copy(TRAIN_LOG, os.path.join(rd, "train.log"))
    if os.path.exists(os.path.join(CONTENT, "build.log")):
        shutil.copy(os.path.join(CONTENT, "build.log"), os.path.join(rd, "build.log"))
    STATUS["notebook_wall_min"] = round((time.time() - T0) / 60, 1)
    json.dump(STATUS, open(os.path.join(rd, "status.json"), "w"), indent=1)
    print("saved to", rd, ":", sorted(os.listdir(rd)))

try:
    _final_save()
except Exception as e:
    banner(f"final save failed: {e}")
print("\nstage status:")
for k, v in STATUS.items():
    if isinstance(v, dict):
        print(f"  {k:8s} {'ok' if v.get('ok') else 'FAILED'} {v.get('secs', '')}s {v.get('error', '')}")
print(f"notebook wall time: {(time.time() - T0) / 60:.1f} min")

if not LOCAL_TEST:
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print("Drive flushed.")
    except Exception as e:
        print("Drive flush failed:", e)
    if AUTO_DISCONNECT:
        print("Disconnecting the runtime now to stop compute-unit use.")
        from google.colab import runtime
        runtime.unassign()